NERUS

In [ ]:
!pip install -q nerus

In [ ]:
import os
import re
import gzip
import urllib.request
from collections import Counter

import pandas as pd
from nerus import load_nerus

In [ ]:
import os
import urllib.request

NERUS_PATH = "nerus_lenta.conllu.gz"
NERUS_URL = "https://storage.yandexcloud.net/natasha-nerus/data/nerus_lenta.conllu.gz"


In [ ]:
docs = load_nerus(NERUS_PATH)
first_doc = next(docs)
print(first_doc.id)
print(first_doc.sents[0].text)

In [ ]:
import os
import re
import gzip
from collections import Counter
import pandas as pd

NERUS_PATH = "nerus_lenta.conllu.gz"
OUTPUT_CSV = "nerus_no_person_loc_org_candidates.csv"

MAX_SAMPLES_PER_BUCKET = {
    "LOC_ONLY": 30,
    "ORG_ONLY": 30,
    "LOC_ORG": 10,
}

FORBIDDEN_ROLE_WORDS = [
    "президент",
    "директор",
    "министр",
    "гражданин",
    "заявитель",
]

MIN_TOKENS = 8
MIN_CHARS = 50

BAD_SHORT_PATTERNS = [
    "сообщает",
    "сообщил",
    "сообщила",
    "сообщили",
    "передает",
    "передал",
    "передала",
    "передали",
    "отмечает",
    "отмечают",
    "заявил",
    "заявила",
    "заявили",
    "по данным",
    "по информации",
    "как сообщили",
    "об этом сообщает",
    "об этом заявил",
]

NAME_LIKE_PATRONYMIC_RE = re.compile(
    r"\b[А-ЯЁ][а-яё]+(?:ович|евич|ич|овна|евна|ична)\b"
)

INITIALS_NAME_RE = re.compile(
    r"\b[А-ЯЁ]\.\s?[А-ЯЁ]\.\s?[А-ЯЁ][а-яё]+\b|"
    r"\b[А-ЯЁ][а-яё]+\s[А-ЯЁ]\.\s?[А-ЯЁ]\.\b|"
    r"\b[А-ЯЁ]\.\s?[А-ЯЁ][а-яё]+\b|"
    r"\b[А-ЯЁ][а-яё]+\s[А-ЯЁ]\.\b"
)

FULL_NAME_LIKE_RE = re.compile(
    r"\b[А-ЯЁ][а-яё]+(?:-[А-ЯЁ][а-яё]+)?\s+"
    r"[А-ЯЁ][а-яё]+(?:-[А-ЯЁ][а-яё]+)?"
    r"(?:\s+[А-ЯЁ][а-яё]+(?:-[А-ЯЁ][а-яё]+)?)?\b"
)

CAPITALIZED_WORD_RE = re.compile(r"\b[А-ЯЁ][а-яё]+\b")

LATIN_NAME_LIKE_RE = re.compile(
    r"\b[A-Z][a-z]+(?:-[A-Z][a-z]+)?\s+[A-Z][a-z]+(?:-[A-Z][a-z]+)?\b"
)

ORG_LOC_MARKERS = [
    "область", "район", "республика", "край", "улица", "проспект",
    "переулок", "набережная", "шоссе", "площадь", "поселок",
    "город", "село", "деревня", "федерация", "служба", "комитет",
    "министерство", "агентство", "департамент", "суд", "дума",
    "компания", "банк", "университет", "институт", "завод",
    "газета", "новости", "администрация", "правительство",
    "обл", "респ", "мэрия"
]


def normalize_text(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()


def extract_entity_type(tag: str) -> str:
    if not tag or tag == "O":
        return "O"
    if "-" in tag:
        return tag.split("-")[-1]
    return tag


def parse_tag_from_misc(misc: str) -> str:
    if not misc:
        return "O"
    m = re.search(r"Tag=([BIO]-[A-Z]+|[A-Z]+)", misc)
    if m:
        return m.group(1)
    return "O"


def is_too_short(text: str, tokens) -> bool:
    if len(tokens) < MIN_TOKENS:
        return True
    if len(text) < MIN_CHARS:
        return True
    return False


def is_bad_short_news(text: str, tokens) -> bool:
    lower = text.lower()
    if len(tokens) < 10 and any(pattern in lower for pattern in BAD_SHORT_PATTERNS):
        return True
    return False


def looks_like_person_name(text: str) -> bool:
    if NAME_LIKE_PATRONYMIC_RE.search(text):
        return True

    if INITIALS_NAME_RE.search(text):
        return True

    if LATIN_NAME_LIKE_RE.search(text):
        return True

    for match in FULL_NAME_LIKE_RE.finditer(text):
        phrase = match.group(0)
        lowered = phrase.lower()
        if not any(marker in lowered for marker in ORG_LOC_MARKERS):
            return True

    return False


def has_forbidden_role_plus_capitalized(text: str) -> bool:
    lowered = text.lower()

    for role in FORBIDDEN_ROLE_WORDS:
        pattern_after = re.compile(
            rf"\b{role}\b(?:\s+\w+){{0,3}}\s+([А-ЯЁ][а-яё]+(?:\s+[А-ЯЁ][а-яё]+){{0,2}})"
        )
        if pattern_after.search(text):
            return True

        pattern_before = re.compile(
            rf"([А-ЯЁ][а-яё]+(?:\s+[А-ЯЁ][а-яё]+){{0,2}})(?:\s+\w+){{0,3}}\s+\b{role}\b"
        )
        if pattern_before.search(text):
            return True

        idx = lowered.find(role)
        if idx != -1:
            left = text[max(0, idx - 40): idx]
            right = text[idx: min(len(text), idx + 50)]
            if CAPITALIZED_WORD_RE.search(left) or CAPITALIZED_WORD_RE.search(right):
                return True

    return False


def classify_sentence(entity_types):
    entity_types = {t for t in entity_types if t != "O"}

    if not entity_types:
        return None
    if entity_types == {"LOC"}:
        return "LOC_ONLY"
    if entity_types == {"ORG"}:
        return "ORG_ONLY"
    if entity_types == {"LOC", "ORG"}:
        return "LOC_ORG"

    return None


def should_reject_sentence(text: str, tags) -> bool:
    ner_types = {extract_entity_type(tag) for tag in tags}

    if "PER" in ner_types or "PERSON" in ner_types:
        return True

    if looks_like_person_name(text):
        return True

    if has_forbidden_role_plus_capitalized(text):
        return True

    return False


def iter_nerus_sentences(conllu_gz_path: str):
    current_text = None
    current_tags = []
    current_tokens = []

    with gzip.open(conllu_gz_path, "rt", encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")

            if not line.strip():
                if current_text is not None:
                    yield {
                        "text": current_text,
                        "tags": current_tags,
                        "tokens": current_tokens,
                    }
                current_text = None
                current_tags = []
                current_tokens = []
                continue

            if line.startswith("# text = "):
                current_text = line[len("# text = "):].strip()
                continue

            if line.startswith("#"):
                continue

            parts = line.split("\t")
            if len(parts) != 10:
                continue

            token_id, token_text = parts[0], parts[1]
            misc = parts[9]

            if "-" in token_id or "." in token_id:
                continue

            tag = parse_tag_from_misc(misc)
            current_tags.append(tag)
            current_tokens.append(token_text)

        if current_text is not None:
            yield {
                "text": current_text,
                "tags": current_tags,
                "tokens": current_tokens,
            }


def extract_nerus_candidates(conllu_gz_path: str, max_samples_per_bucket=None):
    if max_samples_per_bucket is None:
        max_samples_per_bucket = {}

    rows = []
    seen_texts = set()
    bucket_counter = Counter()

    for sent in iter_nerus_sentences(conllu_gz_path):
        text = normalize_text(sent["text"])
        tags = sent["tags"]
        tokens = sent["tokens"]

        if not text:
            continue

        if text in seen_texts:
            continue

        if is_too_short(text, tokens):
            continue

        if is_bad_short_news(text, tokens):
            continue

        if should_reject_sentence(text, tags):
            continue

        entity_types = {extract_entity_type(tag) for tag in tags}
        bucket = classify_sentence(entity_types)
        if bucket is None:
            continue

        if bucket in max_samples_per_bucket and bucket_counter[bucket] >= max_samples_per_bucket[bucket]:
            continue

        loc_count = sum(1 for t in tags if extract_entity_type(t) == "LOC")
        org_count = sum(1 for t in tags if extract_entity_type(t) == "ORG")

        rows.append({
            "source": "nerus",
            "bucket": bucket,
            "text": text,
            "token_count": len(tokens),
            "char_count": len(text),
            "has_loc": int("LOC" in entity_types),
            "has_org": int("ORG" in entity_types),
            "loc_token_count": loc_count,
            "org_token_count": org_count,
        })

        seen_texts.add(text)
        bucket_counter[bucket] += 1

        if max_samples_per_bucket:
            done = all(
                bucket_counter[b] >= max_samples_per_bucket[b]
                for b in max_samples_per_bucket
            )
            if done:
                break

    return pd.DataFrame(rows)



df = extract_nerus_candidates(
    NERUS_PATH,
    max_samples_per_bucket=MAX_SAMPLES_PER_BUCKET
)

print(len(df))
print(df["bucket"].value_counts(dropna=False).to_dict())

print("\nLOC_ONLY:")
print(df[df["bucket"] == "LOC_ONLY"]["text"].head(20).tolist())

print("\nORG_ONLY:")
print(df[df["bucket"] == "ORG_ONLY"]["text"].head(20).tolist())

print("\nLOC_ORG:")
print(df[df["bucket"] == "LOC_ORG"]["text"].head(20).tolist())

df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

NEREL

In [ ]:
!pip install razdel pandas -q

In [ ]:
!git clone https://github.com/nerel-ds/NEREL.git

In [ ]:
import os
import re
from collections import Counter
import pandas as pd

NEREL_PATH = "/content/NEREL/NEREL-v1.0/train"
OUTPUT_CSV = "nerel_no_person_loc_org_candidates.csv"

MAX_SAMPLES_PER_BUCKET = {
    "LOC_ONLY": 30,
    "ORG_ONLY": 30,
    "LOC_ORG": 10,
}

FORBIDDEN_ROLE_WORDS = [
    "президент",
    "директор",
    "министр",
    "гражданин",
    "заявитель",
]

MIN_TOKENS = 8
MIN_CHARS = 50

BAD_SHORT_PATTERNS = [
    "сообщает",
    "сообщил",
    "сообщила",
    "сообщили",
    "передает",
    "передал",
    "передала",
    "передали",
    "отмечает",
    "отмечают",
    "заявил",
    "заявила",
    "заявили",
    "по данным",
    "по информации",
    "как сообщили",
    "об этом сообщает",
    "об этом заявил",
]

NAME_LIKE_PATRONYMIC_RE = re.compile(
    r"\b[А-ЯЁ][а-яё]+(?:ович|евич|ич|овна|евна|ична)\b"
)

INITIALS_NAME_RE = re.compile(
    r"\b[А-ЯЁ]\.\s?[А-ЯЁ]\.\s?[А-ЯЁ][а-яё]+\b|"
    r"\b[А-ЯЁ][а-яё]+\s[А-ЯЁ]\.\s?[А-ЯЁ]\.\b|"
    r"\b[А-ЯЁ]\.\s?[А-ЯЁ][а-яё]+\b|"
    r"\b[А-ЯЁ][а-яё]+\s[А-ЯЁ]\.\b"
)

FULL_NAME_LIKE_RE = re.compile(
    r"\b[А-ЯЁ][а-яё]+(?:-[А-ЯЁ][а-яё]+)?\s+"
    r"[А-ЯЁ][а-яё]+(?:-[А-ЯЁ][а-яё]+)?"
    r"(?:\s+[А-ЯЁ][а-яё]+(?:-[А-ЯЁ][а-яё]+)?)?\b"
)

CAPITALIZED_WORD_RE = re.compile(r"\b[А-ЯЁ][а-яё]+\b")

LATIN_NAME_LIKE_RE = re.compile(
    r"\b[A-Z][a-z]+(?:-[A-Z][a-z]+)?\s+[A-Z][a-z]+(?:-[A-Z][a-z]+)?\b"
)

ORG_LOC_MARKERS = [
    "область", "район", "республика", "край", "улица", "проспект",
    "переулок", "набережная", "шоссе", "площадь", "поселок",
    "город", "село", "деревня", "федерация", "служба", "комитет",
    "министерство", "агентство", "департамент", "суд", "дума",
    "компания", "банк", "университет", "институт", "завод",
    "газета", "новости", "администрация", "правительство",
    "обл", "респ", "мэрия"
]

LOC_TYPES = {
    "LOC", "LOCATION", "CITY", "COUNTRY", "STATE_OR_PROVINCE",
    "STATE_OR_PROV", "DISTRICT", "REGION", "FACILITY", "GPE"
}
ORG_TYPES = {"ORG", "ORGANIZATION"}
PERSON_TYPES = {"PER", "PERSON"}

def normalize_text(text: str) -> str:
    return re.sub(r"\s+", " ", str(text)).strip()


def tokenize_simple(text: str) -> list[str]:
    return re.findall(r"\w+|[^\w\s]", text, flags=re.UNICODE)


def is_too_short(text: str, tokens: list[str]) -> bool:
    return len(tokens) < MIN_TOKENS or len(text) < MIN_CHARS


def is_bad_short_news(text: str, tokens: list[str]) -> bool:
    lower = text.lower()
    if len(tokens) < 10 and any(pattern in lower for pattern in BAD_SHORT_PATTERNS):
        return True
    return False


def normalize_entity_type(raw_type: str) -> str:
    if raw_type is None:
        return "O"

    t = str(raw_type).strip().upper()

    if t in PERSON_TYPES:
        return "PER"
    if t in ORG_TYPES:
        return "ORG"
    if t in LOC_TYPES:
        return "LOC"

    return t


def looks_like_person_name(text: str) -> bool:
    if NAME_LIKE_PATRONYMIC_RE.search(text):
        return True

    if INITIALS_NAME_RE.search(text):
        return True

    if LATIN_NAME_LIKE_RE.search(text):
        return True

    for match in FULL_NAME_LIKE_RE.finditer(text):
        phrase = match.group(0)
        lowered = phrase.lower()
        if not any(marker in lowered for marker in ORG_LOC_MARKERS):
            return True

    return False


def has_forbidden_role_plus_capitalized(text: str) -> bool:
    lowered = text.lower()

    for role in FORBIDDEN_ROLE_WORDS:
        pattern_after = re.compile(
            rf"\b{role}\b(?:\s+\w+){{0,3}}\s+([А-ЯЁ][а-яё]+(?:\s+[А-ЯЁ][а-яё]+){{0,2}})"
        )
        if pattern_after.search(text):
            return True

        pattern_before = re.compile(
            rf"([А-ЯЁ][а-яё]+(?:\s+[А-ЯЁ][а-яё]+){{0,2}})(?:\s+\w+){{0,3}}\s+\b{role}\b"
        )
        if pattern_before.search(text):
            return True

        idx = lowered.find(role)
        if idx != -1:
            left = text[max(0, idx - 40): idx]
            right = text[idx:min(len(text), idx + 50)]
            if CAPITALIZED_WORD_RE.search(left) or CAPITALIZED_WORD_RE.search(right):
                return True

    return False


def classify_sentence(entity_types: set[str]):
    entity_types = {t for t in entity_types if t != "O"}

    if not entity_types:
        return None
    if entity_types == {"LOC"}:
        return "LOC_ONLY"
    if entity_types == {"ORG"}:
        return "ORG_ONLY"
    if entity_types == {"LOC", "ORG"}:
        return "LOC_ORG"
    return None


def should_reject_sentence(text: str, entity_types: set[str]) -> bool:
    if "PER" in entity_types:
        return True

    if looks_like_person_name(text):
        return True

    if has_forbidden_role_plus_capitalized(text):
        return True

    return False


def parse_brat_ann(ann_path: str) -> list[dict]:
    entities = []

    with open(ann_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or not line.startswith("T"):
                continue

            parts = line.split("\t")
            if len(parts) < 3:
                continue

            ent_id = parts[0]
            meta = parts[1]
            ent_text = parts[2]

            meta_parts = meta.split()
            if len(meta_parts) < 3:
                continue

            ent_type = meta_parts[0]
            span_str = " ".join(meta_parts[1:])

            spans = []
            for chunk in span_str.split(";"):
                chunk = chunk.strip()
                chunk_parts = chunk.split()
                if len(chunk_parts) < 2:
                    continue
                try:
                    start = int(chunk_parts[0])
                    end = int(chunk_parts[1])
                    spans.append((start, end))
                except ValueError:
                    continue

            if not spans:
                continue

            start = min(s for s, _ in spans)
            end = max(e for _, e in spans)

            entities.append({
                "id": ent_id,
                "type": ent_type,
                "start": start,
                "end": end,
                "text": ent_text,
                "spans": spans,
            })

    return entities

SENT_BOUNDARY_RE = re.compile(r'(?<=[.!?])[\s\n]+')

def split_text_into_sentences_with_offsets(text: str):
    sentences = []
    start = 0

    for match in SENT_BOUNDARY_RE.finditer(text):
        end = match.start()
        sent_text = text[start:end].strip()
        if sent_text:
            real_start = start
            while real_start < len(text) and text[real_start].isspace():
                real_start += 1
            real_end = end
            while real_end > real_start and text[real_end - 1].isspace():
                real_end -= 1

            if real_end > real_start:
                sentences.append({
                    "text": text[real_start:real_end],
                    "start": real_start,
                    "end": real_end,
                })
        start = match.end()

    tail = text[start:].strip()
    if tail:
        real_start = start
        while real_start < len(text) and text[real_start].isspace():
            real_start += 1
        real_end = len(text)
        while real_end > real_start and text[real_end - 1].isspace():
            real_end -= 1

        if real_end > real_start:
            sentences.append({
                "text": text[real_start:real_end],
                "start": real_start,
                "end": real_end,
            })

    return sentences


def entity_belongs_to_sentence(entity: dict, sent_start: int, sent_end: int) -> bool:
    return entity["start"] >= sent_start and entity["end"] <= sent_end


def iter_nerel_brat_sentences(folder_path: str):
    all_files = os.listdir(folder_path)
    txt_files = sorted([f for f in all_files if f.endswith(".txt")])

    for txt_file in txt_files:
        base = os.path.splitext(txt_file)[0]
        ann_file = base + ".ann"

        txt_path = os.path.join(folder_path, txt_file)
        ann_path = os.path.join(folder_path, ann_file)

        if not os.path.exists(ann_path):
            continue

        with open(txt_path, "r", encoding="utf-8") as f:
            full_text = f.read()

        entities = parse_brat_ann(ann_path)
        sentences = split_text_into_sentences_with_offsets(full_text)

        for i, sent in enumerate(sentences):
            sent_entities = [
                ent for ent in entities
                if entity_belongs_to_sentence(ent, sent["start"], sent["end"])
            ]

            yield {
                "doc_id": base,
                "sent_id": f"{base}_{i}",
                "text": sent["text"],
                "start": sent["start"],
                "end": sent["end"],
                "entities": sent_entities,
            }


def extract_entities_for_text(record: dict):
    text = record.get("text")
    if not text:
        return None, []

    entities = record.get("entities", [])
    entity_types = []

    for ent in entities:
        ent_type = normalize_entity_type(ent.get("type"))
        if ent_type != "O":
            entity_types.append(ent_type)

    return text, entity_types


def inspect_nerel_sample(path: str, n: int = 5):

    for i, rec in enumerate(iter_nerel_brat_sentences(path)):
        if i >= n:
            break

        print("doc_id:", rec["doc_id"])
        print("sent_id:", rec["sent_id"])
        print("text:", rec["text"])
        print("entities:", [
            {
                "type": e["type"],
                "text": e["text"],
                "span": (e["start"], e["end"])
            }
            for e in rec["entities"]
        ])
        print()


def extract_nerel_candidates(path: str, max_samples_per_bucket=None) -> pd.DataFrame:
    if max_samples_per_bucket is None:
        max_samples_per_bucket = {}

    rows = []
    seen_texts = set()
    bucket_counter = Counter()

    for record in iter_nerel_brat_sentences(path):
        text, entity_types_list = extract_entities_for_text(record)
        if not text:
            continue

        text = normalize_text(text)
        tokens = tokenize_simple(text)

        if not text:
            continue

        if text in seen_texts:
            continue

        if is_too_short(text, tokens):
            continue

        if is_bad_short_news(text, tokens):
            continue

        entity_types = set(entity_types_list)

        if should_reject_sentence(text, entity_types):
            continue

        bucket = classify_sentence(entity_types)
        if bucket is None:
            continue

        if bucket in max_samples_per_bucket and bucket_counter[bucket] >= max_samples_per_bucket[bucket]:
            continue

        loc_count = sum(1 for t in entity_types_list if t == "LOC")
        org_count = sum(1 for t in entity_types_list if t == "ORG")

        rows.append({
            "source": "nerel",
            "doc_id": record.get("doc_id"),
            "sent_id": record.get("sent_id"),
            "bucket": bucket,
            "text": text,
            "token_count": len(tokens),
            "char_count": len(text),
            "has_loc": int("LOC" in entity_types),
            "has_org": int("ORG" in entity_types),
            "loc_entity_count": loc_count,
            "org_entity_count": org_count,
        })

        seen_texts.add(text)
        bucket_counter[bucket] += 1

        if max_samples_per_bucket:
            done = all(
                bucket_counter[b] >= max_samples_per_bucket[b]
                for b in max_samples_per_bucket
            )
            if done:
                break

    return pd.DataFrame(rows)


if __name__ == "__main__":

    inspect_nerel_sample(NEREL_PATH, n=5)

    df = extract_nerel_candidates(
        NEREL_PATH,
        max_samples_per_bucket=MAX_SAMPLES_PER_BUCKET
    )

    print("Всего отобрано:", len(df))

    if not df.empty:
        print(df["bucket"].value_counts(dropna=False).to_dict())

        print("\nLOC_ONLY:")
        print(df[df["bucket"] == "LOC_ONLY"]["text"].head(20).tolist())

        print("\nORG_ONLY:")
        print(df[df["bucket"] == "ORG_ONLY"]["text"].head(20).tolist())

        print("\nLOC_ORG:")
        print(df[df["bucket"] == "LOC_ORG"]["text"].head(20).tolist())

        df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")


FACT-RU-EVAL

In [ ]:
!pip install datasets pandas -q

import re
from collections import Counter

import pandas as pd
from datasets import load_dataset, concatenate_datasets

DATASET_NAME = "gusevski/factrueval2016"
OUTPUT_CSV = "factrueval2016_no_person_loc_org_candidates.csv"

In [ ]:
MAX_SAMPLES_PER_BUCKET = {
    "LOC_ONLY": 30,
    "ORG_ONLY": 30,
    "LOC_ORG": 10,
}

FORBIDDEN_ROLE_WORDS = [
    "президент",
    "директор",
    "министр",
    "гражданин",
    "заявитель",
]

MIN_TOKENS = 16
MIN_CHARS = 100

BAD_SHORT_PATTERNS = [
    "сообщает",
    "сообщил",
    "сообщила",
    "сообщили",
    "передает",
    "передал",
    "передала",
    "передали",
    "отмечает",
    "отмечают",
    "заявил",
    "заявила",
    "заявили",
    "по данным",
    "по информации",
    "как сообщили",
    "об этом сообщает",
    "об этом заявил",
]

NAME_LIKE_PATRONYMIC_RE = re.compile(
    r"\b[А-ЯЁ][а-яё]+(?:ович|евич|ич|овна|евна|ична)\b"
)

INITIALS_NAME_RE = re.compile(
    r"\b[А-ЯЁ]\.\s?[А-ЯЁ]\.\s?[А-ЯЁ][а-яё]+\b|"
    r"\b[А-ЯЁ][а-яё]+\s[А-ЯЁ]\.\s?[А-ЯЁ]\.\b|"
    r"\b[А-ЯЁ]\.\s?[А-ЯЁ][а-яё]+\b|"
    r"\b[А-ЯЁ][а-яё]+\s[А-ЯЁ]\.\b|"
    r"\b[А-ЯЁ]\.[А-ЯЁ]\.\s?[А-ЯЁ][а-яё]+\b|"
    r"\b[А-ЯЁ]\.[А-ЯЁ][а-яё]+\b"
)

FULL_NAME_LIKE_RE = re.compile(
    r"\b[А-ЯЁ][а-яё]+(?:-[А-ЯЁ][а-яё]+)?\s+"
    r"[А-ЯЁ][а-яё]+(?:-[А-ЯЁ][а-яё]+)?"
    r"(?:\s+[А-ЯЁ][а-яё]+(?:-[А-ЯЁ][а-яё]+)?)?\b"
)

CAPITALIZED_WORD_RE = re.compile(r"\b[А-ЯЁ][а-яё]+\b")

LATIN_NAME_LIKE_RE = re.compile(
    r"\b[A-Z][a-z]+(?:-[A-Z][a-z]+)?\s+[A-Z][a-z]+(?:-[A-Z][a-z]+)?\b"
)

ORG_LOC_MARKERS = [
    "область", "район", "республика", "край", "улица", "проспект",
    "переулок", "набережная", "шоссе", "площадь", "поселок",
    "город", "село", "деревня", "федерация", "служба", "комитет",
    "министерство", "агентство", "департамент", "суд", "дума",
    "компания", "банк", "университет", "институт", "завод",
    "газета", "новости", "администрация", "правительство",
    "обл", "респ", "мэрия"
]

LOC_LABEL_KEYS = {"LOC", "B-LOC", "I-LOC", "LOCATION", "B-LOCATION", "I-LOCATION"}
ORG_LABEL_KEYS = {"ORG", "B-ORG", "I-ORG", "ORGANIZATION", "B-ORGANIZATION", "I-ORGANIZATION"}
PER_LABEL_KEYS = {"PER", "B-PER", "I-PER", "PERSON", "B-PERSON", "I-PERSON"}


def normalize_text(text: str) -> str:
    text = re.sub(r"\s+", " ", str(text)).strip()
    text = re.sub(r"\s+([,.;:!?])", r"\1", text)
    text = re.sub(r"([(\[]) +", r"\1", text)
    text = re.sub(r" +([)\]])", r"\1", text)
    return text


def join_tokens(tokens: list[str]) -> str:
    if not tokens:
        return ""

    text = " ".join(map(str, tokens))

    text = re.sub(r"\s+([,.;:!?%])", r"\1", text)

    text = re.sub(r"\(\s+", "(", text)
    text = re.sub(r"\s+\)", ")", text)
    text = re.sub(r"\[\s+", "[", text)
    text = re.sub(r"\s+\]", "]", text)

    text = text.replace(" ''", "''")
    text = text.replace(" ``", "``")
    text = text.replace(" ,", ",")
    text = text.replace(" .", ".")
    text = text.replace(" !", "!")
    text = text.replace(" ?", "?")
    text = text.replace(" :", ":")
    text = text.replace(" ;", ";")
    text = text.replace(" %", "%")

    return normalize_text(text)


def tokenize_simple(text: str) -> list[str]:
    return re.findall(r"\w+|[^\w\s]", text, flags=re.UNICODE)


def is_too_short(text: str, tokens: list[str]) -> bool:
    return len(tokens) < MIN_TOKENS or len(text) < MIN_CHARS


def is_bad_short_news(text: str, tokens: list[str]) -> bool:
    lower = text.lower()
    if len(tokens) < 20 and any(pattern in lower for pattern in BAD_SHORT_PATTERNS):
        return True
    return False


def looks_like_person_name(text: str) -> bool:
    if NAME_LIKE_PATRONYMIC_RE.search(text):
        return True

    if INITIALS_NAME_RE.search(text):
        return True

    if LATIN_NAME_LIKE_RE.search(text):
        return True

    for match in FULL_NAME_LIKE_RE.finditer(text):
        phrase = match.group(0)
        lowered = phrase.lower()
        if not any(marker in lowered for marker in ORG_LOC_MARKERS):
            return True

    return False


def has_forbidden_role_plus_capitalized(text: str) -> bool:
    lowered = text.lower()

    for role in FORBIDDEN_ROLE_WORDS:
        pattern_after = re.compile(
            rf"\b{role}\b(?:\s+\w+){{0,3}}\s+([А-ЯЁ][а-яё]+(?:\s+[А-ЯЁ][а-яё]+){{0,2}})"
        )
        if pattern_after.search(text):
            return True

        pattern_before = re.compile(
            rf"([А-ЯЁ][а-яё]+(?:\s+[А-ЯЁ][а-яё]+){{0,2}})(?:\s+\w+){{0,3}}\s+\b{role}\b"
        )
        if pattern_before.search(text):
            return True

        idx = lowered.find(role)
        if idx != -1:
            left = text[max(0, idx - 50): idx]
            right = text[idx: min(len(text), idx + 60)]
            if CAPITALIZED_WORD_RE.search(left) or CAPITALIZED_WORD_RE.search(right):
                return True

    return False


def normalize_tag_to_entity_type(tag_value) -> str:
    if tag_value is None:
        return "O"

    tag_str = str(tag_value).upper()

    if tag_str in PER_LABEL_KEYS:
        return "PER"
    if tag_str in LOC_LABEL_KEYS:
        return "LOC"
    if tag_str in ORG_LABEL_KEYS:
        return "ORG"
    if tag_str == "O":
        return "O"

    if "PER" in tag_str:
        return "PER"
    if "LOC" in tag_str:
        return "LOC"
    if "ORG" in tag_str:
        return "ORG"

    return "O"


def should_reject_sentence(text: str, entity_types: set[str]) -> bool:
    if "PER" in entity_types:
        return True

    if looks_like_person_name(text):
        return True

    if has_forbidden_role_plus_capitalized(text):
        return True

    return False


def classify_sentence(entity_types: set[str]):
    entity_types = {t for t in entity_types if t != "O"}

    if not entity_types:
        return None
    if entity_types == {"LOC"}:
        return "LOC_ONLY"
    if entity_types == {"ORG"}:
        return "ORG_ONLY"
    if entity_types == {"LOC", "ORG"}:
        return "LOC_ORG"
    return None


def merge_all_splits(dataset_dict):
    split_names = list(dataset_dict.keys())
    datasets_list = [dataset_dict[s] for s in split_names]
    merged = concatenate_datasets(datasets_list)
    return merged, split_names


def inspect_dataset_schema(ds, n_outer=3, n_inner=3):
    print("Columns:", ds.column_names)
    print("Num outer rows:", len(ds))

    for i in range(min(n_outer, len(ds))):
        row = ds[i]
        print(f"\n--- OUTER SAMPLE {i} ---")
        print("keys:", list(row.keys()))

        data = row.get("data", [])
        print("inner items:", len(data))

        for j, item in enumerate(data[:n_inner]):
            print("  keys:", list(item.keys()))
            print("  tokens:", item.get("tokens", [])[:20])
            print("  ner_tags_str:", item.get("ner_tags_str", [])[:20])
            print("  ner_tags:", item.get("ner_tags", [])[:20])


def iter_factrueval_sentences(ds):
    for outer_idx, row in enumerate(ds):
        data = row.get("data", [])
        if not isinstance(data, list):
            continue

        for inner_idx, item in enumerate(data):
            if not isinstance(item, dict):
                continue

            tokens = item.get("tokens", [])
            ner_tags_str = item.get("ner_tags_str", [])
            ner_tags = item.get("ner_tags", [])

            if not isinstance(tokens, list) or len(tokens) == 0:
                continue

            if isinstance(ner_tags_str, list) and len(ner_tags_str) == len(tokens):
                tags = ner_tags_str
            elif isinstance(ner_tags, list) and len(ner_tags) == len(tokens):
                tags = ner_tags
            else:
                tags = []

            text = join_tokens(tokens)

            yield {
                "outer_id": outer_idx,
                "inner_id": inner_idx,
                "item_id": item.get("id"),
                "tokens": tokens,
                "text": text,
                "tags": tags,
            }

def extract_factrueval_candidates(ds, max_samples_per_bucket=None) -> pd.DataFrame:
    if max_samples_per_bucket is None:
        max_samples_per_bucket = {}

    rows = []
    seen_texts = set()
    bucket_counter = Counter()

    for sent in iter_factrueval_sentences(ds):
        text = normalize_text(sent["text"])
        tokens = sent["tokens"]
        raw_tags = sent["tags"]

        if not text:
            continue

        if text in seen_texts:
            continue

        simple_tokens = tokenize_simple(text)

        if is_too_short(text, simple_tokens):
            continue

        if is_bad_short_news(text, simple_tokens):
            continue

        entity_types_list = [normalize_tag_to_entity_type(tag) for tag in raw_tags]
        entity_types = set(entity_types_list)

        if should_reject_sentence(text, entity_types):
            continue

        bucket = classify_sentence(entity_types)
        if bucket is None:
            continue

        if bucket in max_samples_per_bucket and bucket_counter[bucket] >= max_samples_per_bucket[bucket]:
            continue

        loc_count = sum(1 for t in entity_types_list if t == "LOC")
        org_count = sum(1 for t in entity_types_list if t == "ORG")

        rows.append({
            "source": "factrueval2016",
            "outer_row_id": sent["outer_id"],
            "inner_row_id": sent["inner_id"],
            "item_id": sent["item_id"],
            "bucket": bucket,
            "text": text,
            "token_count": len(simple_tokens),
            "char_count": len(text),
            "has_loc": int("LOC" in entity_types),
            "has_org": int("ORG" in entity_types),
            "loc_entity_count": loc_count,
            "org_entity_count": org_count,
        })

        seen_texts.add(text)
        bucket_counter[bucket] += 1

        if max_samples_per_bucket:
            done = all(
                bucket_counter[b] >= max_samples_per_bucket[b]
                for b in max_samples_per_bucket
            )
            if done:
                break

    return pd.DataFrame(rows)


dataset = load_dataset(DATASET_NAME)


merged_ds, split_names = merge_all_splits(dataset)


inspect_dataset_schema(merged_ds, n_outer=2, n_inner=3)

df = extract_factrueval_candidates(
    merged_ds,
    max_samples_per_bucket=MAX_SAMPLES_PER_BUCKET
)

print("\nВсего отобрано:", len(df))

if not df.empty:
    print(df["bucket"].value_counts(dropna=False).to_dict())

    print("\nLOC_ONLY:")
    print(df[df["bucket"] == "LOC_ONLY"]["text"].head(20).tolist())

    print("\nORG_ONLY:")
    print(df[df["bucket"] == "ORG_ONLY"]["text"].head(20).tolist())

    print("\nLOC_ORG:")
    print(df[df["bucket"] == "LOC_ORG"]["text"].head(20).tolist())

    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
else:
    print("- MIN_TOKENS = 12")
    print("- MIN_CHARS = 80")